In [0]:
# Test-only drift injection (same idea as the fraud reference): score a normal "yesterday" window and a
# deliberately shifted "today" window with the champion model and append both to the inference log,
# so refresh_monitor -> check_drift -> drift_gate -> retrain can be exercised end to end.
# Refuses to run in PROD.
import sys
import os
sys.path.insert(0, os.getcwd())

import mlflow
from mlflow import MlflowClient
from pyspark.sql import functions as F
from utils import FEATURE_COLS, load_project, model_name as qualified_model_name, table

mlflow.set_registry_uri("databricks-uc")
cfg = load_project()

dbutils.widgets.text("catalog", "stock_demo_test")
dbutils.widgets.text("target", "test")
dbutils.widgets.text("sample_rows", "5000")
catalog = dbutils.widgets.get("catalog")
target = dbutils.widgets.get("target")
sample_rows = int(dbutils.widgets.get("sample_rows"))

if target == "prod" or catalog.endswith("_prod"):
    dbutils.notebook.exit("refused: never inject synthetic drift into PROD")

name = qualified_model_name(cfg, catalog)
model_version = str(MlflowClient().get_model_version_by_alias(name, "champion").version)
model = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{name}@champion", result_type="double")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.serving")
df = spark.table(table(cfg, "features", catalog))
for c in FEATURE_COLS:
    df = df.withColumn(c, F.col(c).cast("double"))
sample = df.orderBy(F.desc("trade_ts")).limit(sample_rows)
KEEP = ["symbol", "trade_ts", *FEATURE_COLS]


def score(frame, request_ts):
    return (frame
            .withColumn("prediction", model(*[F.col(c) for c in FEATURE_COLS]))
            .withColumn("actual_next_close", F.col("target_next_close").cast("double"))
            .withColumn("request_ts", request_ts)
            .withColumn("model_version", F.lit(model_version))
            .withColumn("model_alias", F.lit("champion"))
            .withColumn("run_source", F.lit("simulated"))
            .select(*KEEP, "prediction", "actual_next_close", "request_ts", "model_version",
                    "model_alias", "run_source"))


# BASELINE window ("yesterday"): normal, unmodified data
baseline = score(sample, F.date_sub(F.current_timestamp(), 1))

# DRIFTED window ("today"): returns shifted up, volume x4, short-term volatility x3
drifted = score(
    sample
    .withColumn("return_1d", F.col("return_1d") + F.lit(0.08))
    .withColumn("volume", F.col("volume") * F.lit(4.0))
    .withColumn("volatility_5", F.col("volatility_5") * F.lit(3.0)),
    F.current_timestamp())

log_table = table(cfg, "inference_log", catalog)
baseline.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(log_table)
drifted.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(log_table)
spark.sql(f"ALTER TABLE {log_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Logged {baseline.count()} baseline rows (yesterday) and {drifted.count()} drifted rows (today) to {log_table}")
print("Now run the monitoring job (refresh_monitor -> check_drift) and expect drift_detected=true.")